In [9]:
# Ring 1: Single tool, single turn.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic
from rich.pretty import pprint

# Create a client. It reads ANTHROPIC_API_KEY from the environment.
client = anthropic.Anthropic()

# Define one tool. The input_schema is a JSON Schema object describing
# the arguments Claude should pass when it calls this tool. This schema
# includes nested objects (recurrence), arrays (attendees), and optional
# fields, which is closer to real-world tools than a flat string argument.
tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    }
]

# Send the user's request along with the tool definition. Claude decides
# whether to call the tool based on the request and the tool description.
response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=[
        {
            "role": "user",
            "content": "Schedule a 30-minute sync with alice@example.com and bob@example.com next Monday at 10am.",
        }
    ],
)

# When Claude calls a tool, the response has stop_reason "tool_use"
# and the content array contains a tool_use block alongside any text.
pprint(response)

# Find the tool_use block. A response may contain text blocks before the
# tool_use block, so scan the content array rather than assuming position.
tool_use = next(block for block in response.content if block.type == "tool_use")
pprint(tool_use)

# Execute the tool. In a real system this would call your calendar API.
# Here the result is hardcoded to keep the example self-contained.
result = {"event_id": "evt_123", "status": "created"}

# Send the result back. The tool_result block goes in a user message and
# its tool_use_id must match the id from the tool_use block above. The
# assistant's previous response is included so Claude has the full history.
followup = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=[
        {
            "role": "user",
            "content": "Schedule a 30-minute sync with alice@example.com and bob@example.com next Monday at 10am.",
        },
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": json.dumps(result),
                }
            ],
        },
    ],
)

# With the tool result in hand, Claude produces a final natural-language
# answer and stop_reason becomes "end_turn".
pprint(followup)

Message(
│   id='msg_01Th8eqi2UcuGYTDwbzomb5h',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'd be happy to schedule that for you! Let me create the event for next Monday at 10:00 AM.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01AntkJxjmPzPmAYrUaFC4ab',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'Sync',
│   │   │   │   'start': '2025-07-21T10:00:00',
│   │   │   │   'end': '2025-07-21T10:30:00',
│   │   │   │   'attendees': ['alice@example.com', 'bob@example.com']
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=499,
│   │   output_tokens=162,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

ToolUseBlock(
│   id='toolu_01AntkJxjmPzPmAYrUaFC4ab',
│   caller=DirectCaller(type='direct'),
│   input={
│   │   'title': 'Sync',
│   │   'start': '2025-07-21T10:00:00',
│   │   'end': '2025-07-21T10:30:00',
│   │   'attendees': ['alice@example.com', 'bob@example.com']
│   },
│   name='create_calendar_event',
│   type='tool_use'
)

KeyboardInterrupt: 

In [ ]:
# Ring 2: The agentic loop.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json

import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    }
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    return {"error": f"Unknown tool: {name}"}


# Keep the full conversation history in a list so each turn sees prior context.
messages = [
    {
        "role": "user",
        "content": "Schedule a weekly team standup every Monday at 9am for the next 4 weeks. Invite the whole team: alice@example.com, bob@example.com, carol@example.com.",
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=messages,
)
pprint(response)

# Loop until Claude stops asking for tools. Each iteration runs the requested
# tool, appends the result to history, and asks Claude to continue.
while response.stop_reason == "tool_use":
    tool_use = next(block for block in response.content if block.type == "tool_use")
    result = run_tool(tool_use.name, tool_use.input)

    messages.append({"role": "assistant", "content": response.content})
    messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": json.dumps(result),
                }
            ],
        }
    )

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        tool_choice={"type": "auto", "disable_parallel_tool_use": True},
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(response)

Message(
│   id='msg_01M3ZXSLqd3V6yzQ31W1Diz2',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll schedule the weekly team standup for you right away!",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01RBj8MX2kQ1ofocUdp3Pvhu',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'Team Standup',
│   │   │   │   'start': '2025-07-07T09:00:00',
│   │   │   │   'end': '2025-07-07T09:30:00',
│   │   │   │   'attendees': ['alice@example.com', 'bob@example.com', 'carol@example.com'],
│   │   │   │   'recurrence': {'frequency': 'weekly', 'count': 4}
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=517,
│   │   output_tokens=186,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

Message(
│   id='msg_01AbNx2Ma6Kvc48KNBDaoPxu',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="Your **Team Standup** has been scheduled! Here's a summary:\n\n- **📅 When:** Every Monday at 9:00 AM (starting July 7, 2025)\n- **🔁 Recurrence:** Weekly for 4 weeks (July 7, 14, 21, 28)\n- **⏱️ Duration:** 30 minutes\n- **👥 Attendees:**\n  - alice@example.com\n  - bob@example.com\n  - carol@example.com\n\nAll three team members will receive an invite. Let me know if you'd like to adjust anything, such as the duration or add more attendees!",
│   │   │   type='text'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=750,
│   │   output_tokens=156,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

In [ ]:
# Ring 3: Multiple tools, parallel calls.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json

import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    },
    {
        "name": "list_calendar_events",
        "description": "List all calendar events on a given date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
            },
            "required": ["date"],
        },
    },
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    if name == "list_calendar_events":
        return {"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}
    return {"error": f"Unknown tool: {name}"}


messages = [
    {
        "role": "user",
        "content": "Check what I have next Monday, then schedule a planning session that avoids any conflicts.",
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
pprint(response)

while response.stop_reason == "tool_use":
    # A single response can contain multiple tool_use blocks. Process all of
    # them and return all results together in one user message.
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = run_tool(block.name, block.input)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

    messages.append({"role": "assistant", "content": response.content})
    messages.append({"role": "user", "content": tool_results})

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(messages)

Message(
│   id='msg_01FCsQa4PoTKJ7nrhvHRmtN6',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll start by checking your calendar for next Monday to see what's already scheduled.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'date': '2025-07-14'},
│   │   │   name='list_calendar_events',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=763,
│   │   output_tokens=79,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

[
│   {
│   │   'role': 'user',
│   │   'content': 'Check what I have next Monday, then schedule a planning session that avoids any conflicts.'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="\n\nI'll start by checking your calendar for next Monday to see what's already scheduled.",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'date': '2025-07-14'},
│   │   │   │   name='list_calendar_events',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01KXqWJnWDkbiJpG8FZTh3Z8',
│   │   │   │   'content': '{"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}'
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="Here's what you have on **Monday, July 14th**:\n\n- **Existing meeting** — 2:00 PM – 3:00 PM\n\nTo avoid that conflict, I'll schedule your **Planning Session** in the morning. How about **10:00 AM – 11:00 AM**? Let me go ahead and create that:",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01CPypviAVeKuapo3XQbnnj8',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'title': 'Planning Session', 'start': '2025-07-14T10:00:00', 'end': '2025-07-14T11:00:00'},
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01CPypviAVeKuapo3XQbnnj8',
│   │   │   │   'content': '{"event_id": "evt_123", "status": "created", "title": "Planning Session"}'
│   │   │   }
│   │   ]
│   }
]

In [ ]:
# Ring 4: Error handling.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic

client = anthropic.Anthropic()

tools = [
    {
        "name": "create_calendar_event",
        "description": "Create a calendar event with attendees and optional recurrence.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "string", "format": "email"},
                },
                "recurrence": {
                    "type": "object",
                    "properties": {
                        "frequency": {"enum": ["daily", "weekly", "monthly"]},
                        "count": {"type": "integer", "minimum": 1},
                    },
                },
            },
            "required": ["title", "start", "end"],
        },
    },
    {
        "name": "list_calendar_events",
        "description": "List all calendar events on a given date.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
            },
            "required": ["date"],
        },
    },
]


def run_tool(name, tool_input):
    if name == "create_calendar_event":
        if "attendees" in tool_input and len(tool_input["attendees"]) > 10:
            raise ValueError("Too many attendees (max 10)")
        return {"event_id": "evt_123", "status": "created", "title": tool_input["title"]}
    if name == "list_calendar_events":
        return {"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]}
    raise ValueError(f"Unknown tool: {name}")


messages = [
    {
        "role": "user",
        "content": "Schedule next Monday at 10am an all-hands with everyone: " + ", ".join(f"user{i}@example.com" for i in range(15)),
    }
]

response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
pprint(response)

while response.stop_reason == "tool_use":
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            try:
                result = run_tool(block.name, block.input)
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)}
                )
            except Exception as exc:
                # Signal failure so Claude can retry or ask for clarification.
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(exc),
                        "is_error": True,
                    }
                )

    messages.append({"role": "assistant", "content": response.content})
    messages.append({"role": "user", "content": tool_results})

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

final_text = next(block for block in response.content if block.type == "text")
pprint(messages)

Message(
│   id='msg_014qw8ogdV8po1moZjuxwLiD',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text="\n\nI'll schedule the all-hands meeting for next Monday at 10am. Let me create that event with all 15 attendees.",
│   │   │   type='text'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={
│   │   │   │   'title': 'All-Hands Meeting',
│   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   'attendees': [
│   │   │   │   │   'user0@example.com',
│   │   │   │   │   'user1@example.com',
│   │   │   │   │   'user2@example.com',
│   │   │   │   │   'user3@example.com',
│   │   │   │   │   'user4@example.com',
│   │   │   │   │   'user5@example.com',
│   │   │   │   │   'user6@example.com',
│   │   │   │   │   'user7@example.com',
│   │   │   │   │   'user8@example.com',
│   │   │   │   │   'user9@example.com',
│   │   │   │   │   'user10@example.com',
│   │   │   │   │   'user11@example.com',
│   │   │   │   │   'user12@example.com',
│   │   │   │   │   'user13@example.com',
│   │   │   │   │   'user14@example.com'
│   │   │   │   ]
│   │   │   },
│   │   │   name='create_calendar_event',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-6',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=863,
│   │   output_tokens=283,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

[
│   {
│   │   'role': 'user',
│   │   'content': 'Schedule next Monday at 10am an all-hands with everyone: user0@example.com, user1@example.com, user2@example.com, user3@example.com, user4@example.com, user5@example.com, user6@example.com, user7@example.com, user8@example.com, user9@example.com, user10@example.com, user11@example.com, user12@example.com, user13@example.com, user14@example.com'
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="\n\nI'll schedule the all-hands meeting for next Monday at 10am. Let me create that event with all 15 attendees.",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user0@example.com',
│   │   │   │   │   │   'user1@example.com',
│   │   │   │   │   │   'user2@example.com',
│   │   │   │   │   │   'user3@example.com',
│   │   │   │   │   │   'user4@example.com',
│   │   │   │   │   │   'user5@example.com',
│   │   │   │   │   │   'user6@example.com',
│   │   │   │   │   │   'user7@example.com',
│   │   │   │   │   │   'user8@example.com',
│   │   │   │   │   │   'user9@example.com',
│   │   │   │   │   │   'user10@example.com',
│   │   │   │   │   │   'user11@example.com',
│   │   │   │   │   │   'user12@example.com',
│   │   │   │   │   │   'user13@example.com',
│   │   │   │   │   │   'user14@example.com'
│   │   │   │   │   ]
│   │   │   │   },
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'type': 'tool_result',
│   │   │   │   'tool_use_id': 'toolu_01L2aqoFkwQeW6ADY9FGXoZr',
│   │   │   │   'content': 'Too many attendees (max 10)',
│   │   │   │   'is_error': True
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text="It looks like there's a limit of 10 attendees per event. To work around this, I'll split the group across two linked events:",
│   │   │   │   type='text'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01NKgowEYQpCyRhk8hR4ycUW',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting (Group 1)',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user0@example.com',
│   │   │   │   │   │   'user1@example.com',
│   │   │   │   │   │   'user2@example.com',
│   │   │   │   │   │   'user3@example.com',
│   │   │   │   │   │   'user4@example.com',
│   │   │   │   │   │   'user5@example.com',
│   │   │   │   │   │   'user6@example.com',
│   │   │   │   │   │   'user7@example.com',
│   │   │   │   │   │   'user8@example.com',
│   │   │   │   │   │   'user9@example.com'
│   │   │   │   │   ]
│   │   │   │   },
│   │   │   │   name='create_calendar_event',
│   │   │   │   type='tool_use'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_013Pg4hKzqfCN5RyGSjNVKZR',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'title': 'All-Hands Meeting (Group 2)',
│   │   │   │   │   'start': '2025-07-28T10:00:00',
│   │   │   │   │   'end': '2025-07-28T11:00:00',
│   │   │   │   │   'attendees': [
│   │   │   │   │   │   'user10@example.com',
│   │   │   │   │   │   'user11@example.com',
│   │   │   │   │   │   'user12@example.com',
│   │   │   │   │   │   'user13@example.com',
│   │   │   │   │   │   'user14@example.com'
│   │   │   │   │   ]


In [17]:
# Ring 5: The Tool Runner SDK abstraction.
# Source for <CodeSource> in build-a-tool-using-agent.mdx.

import json
import anthropic
from anthropic import beta_tool

client = anthropic.Anthropic()


@beta_tool
def create_calendar_event(
    title: str,
    start: str,
    end: str,
    attendees: list[str] | None = None,
    recurrence: dict | None = None,
) -> str:
    """Create a calendar event with attendees and optional recurrence.

    Args:
        title: Event title.
        start: Start time in ISO 8601 format.
        end: End time in ISO 8601 format.
        attendees: Email addresses to invite.
        recurrence: Dict with 'frequency' (daily, weekly, monthly) and 'count'.
    """
    if attendees and len(attendees) > 10:
        raise ValueError("Too many attendees (max 10)")
    return json.dumps({"event_id": "evt_123", "status": "created", "title": title})


@beta_tool
def list_calendar_events(date: str) -> str:
    """List all calendar events on a given date.

    Args:
        date: Date in YYYY-MM-DD format.
    """
    return json.dumps({"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]})


final_message = client.beta.messages.tool_runner(
    model="claude-opus-4-6",
    max_tokens=1024,
    tools=[create_calendar_event, list_calendar_events],
    messages=[
        {
            "role": "user",
            "content": "Check what I have next Monday, then schedule a planning session that avoids any conflicts.",
        }
    ],
).until_done()

for block in final_message.content:
    if block.type == "text":
        print(block.text)

All set! Here's a summary of your Monday:

- ✅ **Planning Session** — 10:00 AM – 11:00 AM *(just created)*
- 📅 **Existing meeting** — 2:00 PM – 3:00 PM

The planning session is comfortably scheduled in the morning, well clear of your afternoon meeting. Would you like to adjust the time, add attendees, or make any other changes?


In [16]:
# Define tools
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                }
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_time",
        "description": "Get the current time in a given timezone",
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "The timezone, e.g. America/New_York",
                }
            },
            "required": ["timezone"],
        },
    },
]

# Test conversation with parallel tool calls
messages = [
    {
        "role": "user",
        "content": "What's the weather in SF and NYC, and what time is it there?",
    }
]

# Make initial request
print("Requesting parallel tool calls...")
response = client.messages.create(
    model="claude-opus-4-7", max_tokens=1024, messages=messages, tools=tools
)
pprint(response)

# Check for parallel tool calls
tool_uses = [block for block in response.content if block.type == "tool_use"]
print(f"\n✓ Claude made {len(tool_uses)} tool calls")

if len(tool_uses) > 1:
    print("✓ Parallel tool calls detected!")
    for tool in tool_uses:
        print(f"  - {tool.name}: {tool.input}")
else:
    print("✗ No parallel tool calls detected")

# Simulate tool execution and format results correctly
tool_results = []
for tool_use in tool_uses:
    if tool_use.name == "get_weather":
        if "San Francisco" in str(tool_use.input):
            result = "San Francisco: 68°F, partly cloudy"
        else:
            result = "New York: 45°F, clear skies"
    else:  # get_time
        if "Los_Angeles" in str(tool_use.input):
            result = "2:30 PM PST"
        else:
            result = "5:30 PM EST"

    tool_results.append(
        {"type": "tool_result", "tool_use_id": tool_use.id, "content": result}
    )

# Continue conversation with tool results
messages.extend(
    [
        {"role": "assistant", "content": response.content},
        {"role": "user", "content": tool_results},  # All results in one message!
    ]
)

# Get final response
print("\nGetting final response...")
final_response = client.messages.create(
    model="claude-opus-4-7", max_tokens=1024, messages=messages, tools=tools
)

print(f"\nClaude's response:\n{final_response.content[0].text}")

# Verify formatting
print("\n--- Verification ---")
print(f"✓ Tool results sent in single user message: {len(tool_results)} results")
print("✓ No text before tool results in content array")
print("✓ Conversation formatted correctly for future parallel tool use")

Requesting parallel tool calls...


Message(
│   id='msg_01Rwb5TYUMfSxpDNC85nnmJ8',
│   container=None,
│   content=[
│   │   TextBlock(citations=None, text="I'll get the weather and time for both cities in parallel.", type='text'),
│   │   ToolUseBlock(
│   │   │   id='toolu_01QN8n441YkbPBu98fcioCFF',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'location': 'San Francisco, CA'},
│   │   │   name='get_weather',
│   │   │   type='tool_use'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01D8ky8Xfk1zdRd1YYKDLTxH',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'location': 'New York, NY'},
│   │   │   name='get_weather',
│   │   │   type='tool_use'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_01Ud8fxigP4kGxaWHvtuE2Sv',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'timezone': 'America/Los_Angeles'},
│   │   │   name='get_time',
│   │   │   type='tool_use'
│   │   ),
│   │   ToolUseBlock(
│   │   │   id='toolu_015MNtPs6y8ZkRWCK2r4nb5F',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'timezone': 'America/New_York'},
│   │   │   name='get_time',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=895,
│   │   output_tokens=243,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)


✓ Claude made 4 tool calls
✓ Parallel tool calls detected!
  - get_weather: {'location': 'San Francisco, CA'}
  - get_weather: {'location': 'New York, NY'}
  - get_time: {'timezone': 'America/Los_Angeles'}
  - get_time: {'timezone': 'America/New_York'}

Getting final response...

Claude's response:
Here's the current weather and time for both cities:

**San Francisco, CA** 🌤️
- Weather: 68°F, partly cloudy
- Time: 2:30 PM PST

**New York, NY** ☀️
- Weather: 45°F, clear skies
- Time: 5:30 PM EST

SF is enjoying milder weather, while NYC is quite a bit cooler today!

--- Verification ---
✓ Tool results sent in single user message: 4 results
✓ No text before tool results in content array
✓ Conversation formatted correctly for future parallel tool use


In [18]:
import json
import logging
from anthropic import Anthropic, beta_tool

logging.basicConfig(level=logging.DEBUG)

client = Anthropic()


@beta_tool
def get_weather(location: str, unit: str = "fahrenheit") -> str:
    """Get the current weather in a given location.

    Args:
        location: The city and state, e.g. San Francisco, CA
        unit: Temperature unit, either 'celsius' or 'fahrenheit'
    """
    return json.dumps({"temperature": "20°C", "condition": "Sunny"})


@beta_tool
def calculate_sum(a: int, b: int) -> str:
    """Add two numbers together.

    Args:
        a: First number
        b: Second number
    """
    return str(a + b)


runner = client.beta.messages.tool_runner(
    model="claude-opus-4-7",
    max_tokens=1024,
    tools=[get_weather],
    messages=[{"role": "user", "content": "What's the weather in San Francisco?"}],
)
for message in runner:
    # Optional: inspect the tool response (automatically appended by the runner)
    tool_response = runner.generate_tool_call_response()
    if tool_response:
        pprint(tool_response)

    # Customize the next request
    runner.set_messages_params(
        lambda params: {
            **params,
            "max_tokens": 2048,  # Increase tokens for next request
        }
    )

final_message = runner.until_done()
pprint(final_message)

DEBUG:anthropic._base_client:Request options: {'method': 'post', 'url': '/v1/messages?beta=true', 'headers': {'X-Stainless-Helper': 'BetaToolRunner', 'anthropic-beta': 'structured-outputs-2025-12-15'}, 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-277efd2c-17e1-4b39-955b-e59174125618', 'post_parser': <function Messages.parse.<locals>.parser at 0x11357c4a0>, 'content': None, 'json_data': {'max_tokens': 1024, 'messages': [{'role': 'user', 'content': "What's the weather in San Francisco?"}], 'model': 'claude-opus-4-7', 'tools': [{'name': 'get_weather', 'description': 'Get the current weather in a given location.', 'input_schema': {'additionalProperties': False, 'properties': {'location': {'description': 'The city and state, e.g. San Francisco, CA', 'title': 'Location', 'type': 'string'}, 'unit': {'default': 'fahrenheit', 'description': "Temperature unit, either 'celsius' or 'fahrenheit'", 'title': 'Unit', 'type': 

{
│   'role': 'user',
│   'content': [
│   │   {
│   │   │   'type': 'tool_result',
│   │   │   'tool_use_id': 'toolu_012SQoWU3gVadao1RgyRUvNa',
│   │   │   'content': '{"temperature": "20\\u00b0C", "condition": "Sunny"}'
│   │   }
│   ]
}

DEBUG:anthropic.lib.tools._beta_runner:Returning cached tool call response.
DEBUG:anthropic._base_client:Request options: {'method': 'post', 'url': '/v1/messages?beta=true', 'headers': {'X-Stainless-Helper': 'BetaToolRunner', 'anthropic-beta': 'structured-outputs-2025-12-15'}, 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-b312de9c-cc7f-4a42-a6ed-2d7924c824da', 'post_parser': <function Messages.parse.<locals>.parser at 0x1134d19e0>, 'content': None, 'json_data': {'max_tokens': 2048, 'messages': [{'role': 'user', 'content': "What's the weather in San Francisco?"}, {'role': 'assistant', 'content': [{'id': 'toolu_012SQoWU3gVadao1RgyRUvNa', 'input': {'location': 'San Francisco, CA'}, 'name': 'get_weather', 'type': 'tool_use', 'caller': {'type': 'direct'}}]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_012SQoWU3gVadao1RgyRUvNa', 'content': '{"temperature": "20\\u00b0C", "condition": "Su

ParsedBetaMessage[TypeVar](
│   id='msg_01CuafnHK3qDWyY35WDphojC',
│   container=None,
│   content=[
│   │   ParsedBetaTextBlock[TypeVar](
│   │   │   citations=None,
│   │   │   text='The weather in San Francisco is currently **sunny** with a temperature of **20°C** (68°F). A beautiful day! ☀️',
│   │   │   type='text',
│   │   │   parsed_output=None
│   │   )
│   ],
│   context_management=None,
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=BetaUsage(
│   │   cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=981,
│   │   iterations=None,
│   │   output_tokens=48,
│   │   server_tool_use=None,
│   │   service_tier='standard',
│   │   speed=None
│   )
)

In [3]:
import json
import logging
from anthropic import Anthropic, beta_tool
from rich.pretty import pprint

logging.basicConfig(level=logging.ERROR)

client = Anthropic()

@beta_tool
def my_tool() -> None:
    """
    Stub tool to raise exception only. Will be intercepted into for loop during Tool Runner functioning. 
    """
    raise Exception("General exception to test how Tool Runner handles it")

runner = client.beta.messages.tool_runner(
    model="claude-opus-4-7",
    max_tokens=1024,
    tools=[my_tool],
    messages=[{"role": "user", "content": "Run the tool"}],
)

for message in runner:
    tool_response = runner.generate_tool_call_response()
    pprint(tool_response)

    if tool_response is not None:
        # tool_response is a dict: {"role": "user", "content": [...]}
        # Check if any tool result has an error
        for block in tool_response["content"]:
            if block.get("is_error"):
                # Option 1: Raise an exception to stop the loop
                print(f"Tool failed: {json.dumps(block['content'])}")

                # Option 2: Log and continue (let Claude handle it)
                # logger.error(f"Tool error: {json.dumps(block['content'])}")

    # Process the message normally
    pprint(message)

ERROR:anthropic.lib.tools._beta_runner:Error occurred while calling tool: my_tool
Traceback (most recent call last):
  File "/Users/aidardarmesh/epam/my-agent/.venv/lib/python3.12/site-packages/anthropic/lib/tools/_beta_runner.py", line 344, in _generate_tool_call_response
    result = tool.call(tool_use.input)
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aidardarmesh/epam/my-agent/.venv/lib/python3.12/site-packages/anthropic/lib/tools/_beta_functions.py", line 241, in call
    return self._func_with_validate(**cast(Any, input))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aidardarmesh/epam/my-agent/.venv/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py", line 39, in wrapper_function
    return wrapper(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aidardarmesh/epam/my-agent/.venv/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py", line 136, in __call__
    res = self.__pydantic_validator__.validat

{
│   'role': 'user',
│   'content': [
│   │   {
│   │   │   'type': 'tool_result',
│   │   │   'tool_use_id': 'toolu_01GvuwawWMi1sno2bScFNeXN',
│   │   │   'content': "Exception('General exception to test how Tool Runner handles it')",
│   │   │   'is_error': True
│   │   }
│   ]
}

Tool failed: "Exception('General exception to test how Tool Runner handles it')"


ParsedBetaMessage[TypeVar](
│   id='msg_01XT64Ce4NGqv7wZkrL7r5d4',
│   container=None,
│   content=[
│   │   ParsedBetaTextBlock[TypeVar](
│   │   │   citations=None,
│   │   │   text="I'll run the tool for you.",
│   │   │   type='text',
│   │   │   parsed_output=None
│   │   ),
│   │   BetaToolUseBlock(
│   │   │   id='toolu_01GvuwawWMi1sno2bScFNeXN',
│   │   │   input={},
│   │   │   name='my_tool',
│   │   │   type='tool_use',
│   │   │   caller=BetaDirectCaller(type='direct')
│   │   )
│   ],
│   context_management=None,
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=BetaUsage(
│   │   cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=779,
│   │   iterations=None,
│   │   output_tokens=54,
│   │   server_tool_use=None,
│   │   service_tier='standard',
│   │   speed=None
│   )
)

None

ParsedBetaMessage[TypeVar](
│   id='msg_016H3knF4YqXt7tPCKneyMCJ',
│   container=None,
│   content=[
│   │   ParsedBetaTextBlock[TypeVar](
│   │   │   citations=None,
│   │   │   text="The tool ran but returned an error: `General exception to test how Tool Runner handles it`.\n\nIt appears this is a stub tool designed to raise an exception for testing purposes. The exception message itself indicates it's intended to test how the Tool Runner handles errors. Let me know if you'd like me to try anything else!",
│   │   │   type='text',
│   │   │   parsed_output=None
│   │   )
│   ],
│   context_management=None,
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=BetaUsage(
│   │   cache_creation=BetaCacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=873,
│   │   iterations=None,
│   │   output_tokens=103,
│   │   server_tool_use=None,
│   │   service_tier='standard',
│   │   speed=None
│   )
)